# Statistical Properties of Financial Security Returns

This notebook covers financial returns and their statistical properties. This is an essential building block for more advanced financial analysis. In theory, you should be familiar with most of the definitions and concepts in this notebook, save perhaps for the distributional properties of estimates of means, variances, skew and kurtosis. We will use stock prices and returns as a working example, but you should understand that the same definitions are relevant for other financial security returns such as government and corporate bonds.

This notebook will help you understand:

1. Alternative definitions of stock returns and how to compute these from price-based data sources
2. How to read historical stock price data from Yahoo! finance into a Pandas DataFrame
3. How to make a basic price time series plot
4. How to compute standard descriptive statistics for stock returns
5. What are "moments" of a distribution including mean, variance, skew, and kurtosis -- how these are defined and what information they convey
6. How to estimate the moments mentioned above
7. How to conduct formal statistical inference for the mean stock return

## Background and definitions



We start with some key definitions:  

Price = the price of a traded financial security, such as a stock, bond, ETF, commodity, etc. 

Mathematically, we will denote the price as $P_{t}$, where the t-subscript represents a particular time. So, $P_{1}, P_{2}, P_{3},...$ denotes the price of a particular stock or other security at times 1, 2, 3, and so on. The units of time depend on the setting, and might be days, months, quarters, years, etc. 

Note: if we are considering multiple securities, such as IBM and GE stock, then we might use notation like $P_{IBM,t}$ and $P_{GE,t}$ to denote the prices of IBM and GE, respectively, on date $t$.

We will assume that prices are always positive. This allows us to take the natural logarithm. The log price is denoted $p_{t} \equiv \ln(P_{t})$. The reason for wanting to consider log prices will be explained below.

Some assets pay periodic cash flows to the holder. For example, some stocks pay regular dividends to shareholders, and some bonds pay periodic coupons to the holder. Mathematically, let $D_{t}$ denote the cash received by the holder of a stock (or other asset) at time $t$. 


## Key return definitions
### Periodic return

Period return formula: $R_{t} \equiv \frac{P_{t} + D_{t} - P_{t-1}}{P_{t-1}} = \frac{P_{t} + D_{t}}{P_{t-1}} - 1 = \frac{P_{t} - P_{t-1}}{P_{t-1}} + \frac{D_{t}}{P_{t-1}}$

The third equality shows that the periodic return can be decomposed into two terms: 1) capital gain and 2) dividend yield.

The return $R_{t}$ defined above is sometimes called the *simple net return* whereas $1 + R_{t}$ is the *simple gross return*. As a simple quantitative example, suppose the price goes from 100 to 105 during a period with no dividends. Then the simple gross return is $1+R_{t} = \frac{105}{100} = 1.05$ and the simple net return is $R_{t} = 1.05 - 1 = 0.05$ or 5%. 

*Technical note*: In the periodic return formula above, the component $D_{t}$ should technically equal the cumulative, re-invested value of all cash flows/dividends received by the investor during the holding period. To understand the issue here, imagine computing the periodic return over one year for a corporate bond that pays a quarterly coupon. The $D_{t}$ component in the return formula in this case should equal the re-invested value of all coupons received during that year. Similar comments would apply to the annual return for a stock that pays quarterly dividends. 


### Log return
Suppose for a moment that the dividend is zero ($D_{t} = 0$). Then the log return can be computed as:

$r_{t} \equiv \ln(P_{t}/P_{t-1}) = p_{t} - p_{t-1}$. (Remember that $p_{t} = \ln(P_{t})$.)

*Often, in practice, we can obtain special price series that effectively include paid dividends.* For example, Yahoo! finance provides an "Adjusted close" price series that incorporates the effects of both dividends and potential stock split events. This is convenient because we can then take $D_{t} = 0$ in return computations. It is particularly convenient for log returns, because as shown above, the log return is simply the difference of (dividend-adjusted) log prices.


### Multiperiod returns

What if a stock is held for multiple days, or months? For simplicity, let us assume that the asset does not pay a dividend. (This is not as unrealistic as it might seem -- many publicly listed companies on Nasdaq and even NYSE have not paid a dividend and are unlikely to do so in the near term.)

Without dividends, the one-period gross return is $1 + R_{t} = \frac{P_{t}}{P_{t-1}}$. Now, let us define the $k$-period gross return as the gross return from period $t-k$ to $t$, or $1+R_{t}(k)$ in notation. This is:

$1+R_{t}(k) = (1+R_{t-k+1})(1+R_{t-k+2})...(1+R_{t}) = \frac{P_{t}}{P_{t-k}}$. The net $k$ period return subtracts one from the previous definition.

Multiperiod annual returns are often expressed on an *annualized* basis for interpretation as follows:

Annualized $R_{t}(k) = [\prod_{j=0}^{k-1}(1+R_{t-j})]^{1/k}-1$

Log returns are very convenient in the multiperiod case. By taking the log of the $k$ period gross return formula above, it is straightforward to verify that the $k$-period log return is just the sum of the period-by-period log returns:

$r_{t}(k) = r_{t-k+1} + r_{t-k+2} + ... + r_{t}$

If the period interval is one year, then we can annualize the log-multiyear return by simply averaging the log returns over the years in question.

Finally, the convenience of log returns for the multiperiod case extends to a setting with dividends. However, there is a complication in the sense that the log return is a nonlinear function of log prices and dividends:

$r_{t} = \ln(P_{t} +  D_{t}) - \ln(P_{t-1})$





## Taking it to the data

Let's take the previous concepts to the data by downloading a history of daily returns for a set of stocks. In practice, dividends and potential stock splits complicate the appropriate construction of returns. Historically, Yahoo Finance provided 'adjusted prices' that accounted for these features. However, recently Yahoo Finance moved to a subscription fee model and stopped making these data freely available. As a result, the data we will use come from the Center for Research in Security Prices (CRSP), which is a leading provider of historical stock data for academic research. At the moment, you do not have direct access to CRSP data, but I should be able to get you access through Wharton Research Data Services (WRDS). I will need to set up a class account for you, and this may take a week or two. 

The specific data used in this notebook are posted under the Data folder for the course on Canvas as "StockData_2015_2024.csv". *In order to run the codes below without error, you will need to download and save this data somewhere on your computer, and then change the "data_loc" string in the code cell below to match the location on your machine where the data are stored.*

The code cell below imports the data and performs some basic data manipulation. The result is a Pandas Dataframe that contains, for each date and each ticker symbol, the daily stock return in decimal format. This is a simple return, i.e., $R_{t} = (P_{t} + D_{t} - P_{t-1})/P_{t-1}$. 

The stocks are:

1. AAPL: Apple
2. BA: Boeing
3. CAT: Caterpillar
4. GE: General Electric
5. IBM: International Business Machines
6. SPY: A leading exchange traded fund (ETF) that tracks the S\&P 500 index. You can think of this as a "market" return
7. TSLA: Tesla
8. XOM: Exxon-Mobil


In [ ]:
import numpy as np
import pandas as pd
import datetime as dt
import os 
#Note: Above we are importing certain key python libraries, all of which are fairly standard/common

data_name = "StockData_2015_2024.csv"
#IMPT: set the string below to the location where you have stored the csv file "StockData_2015_2024.csv"
data_loc = r"C:\Users\bpaye\Dropbox\Teaching\Pamplin_FIN4414\Data"

#switch to directory where data is saved
os.chdir(data_loc)
cwd = os.getcwd() 
print("Current working directory:", cwd) 

#Now read the data -- we will use the 'read_csv' feature from Pandas
csvFile = pd.read_csv(data_name)
print(csvFile)

#create a pandas data frame
df = pd.DataFrame(csvFile)
df.head()

print(df.dtypes)

#convert Ticker and date fields to string
df['TICKER'] = df['TICKER'].astype('string')
df['date'] = df['date'].astype('string')

#now convert date from string to 'datetime'
df['date'] = pd.to_datetime(df['date'])

print(df.dtypes)
df.head()

# Reshape to a 'wide' format from the native 'long' format
df_wide = df.pivot(index='date', columns='TICKER', values='RET')
df_wide.head()



Next we will create certain other related datasets that transform the simple returns above. These include:

1. "Gross" returns: this is just one plus the simple return (we'll call this 'df_greturns')
2. Cumulative returns: this is a cumulative version of the multiperiod returns we discussed above. The interpretation is that one can imagine investing \$1 in the stock at the beginning of the sample period. This is the value of that investment at each subsequent date. We'll call it `df_creturns`.
3. Log returns: here we transform the simple return to a logarithmic return using the math expressions above. We will call this `df_logreturns`.

The last statement prints the first few rows of the log returns. Comparing with the simple returns above, we see that the log return is always slightly smaller than the simple return, but the returns are quite similar. The differences are larger when the magnitude of the return is larger.

In [7]:
#rename returns dataset
df_returns = df_wide

#create dataset of gross returns by simply adding one to each return
df_greturns = df_returns + 1

#Now create dataset of cumulative returns by computing the cumulative product for each column
df_creturns = df_greturns.cumprod()

df_logreturns = np.log(df_greturns)
df_logreturns.head()


TICKER,AAPL,BA,CAT,GE,IBM,SPY,TSLA,XOM
date,,,,,,,,
2015-01-02,-0.009559,-0.000231,0.003817,-0.008345,0.010046,-0.000535,-0.014036,0.004102
2015-01-05,-0.028576,-0.006950,-0.054230,-0.018527,-0.015860,-0.018225,-0.042950,-0.027743
2015-01-06,0.000094,-0.011848,-0.006456,-0.021780,-0.021802,-0.009464,0.005648,-0.005330
2015-01-07,0.013925,0.015407,0.015378,0.000415,-0.006557,0.012384,-0.001563,0.010082
2015-01-08,0.037702,0.017527,0.010197,0.011971,0.021502,0.017589,-0.001589,0.016508


Now we have a nice dataframe of daily stock returns. It is good practice to visualize the data to be sure that there are not any weird anomalies. If so, we will want to explore these and determine what is the cause before proceeding with further analyses.

As an example of this type of exploration, let us plot the cumulative series for Exxon-Mobil (XOM). We can do this using Python's 'Matplotlib' functionality. The following code cells make two plots:

1. A basic plot of the cumulative return for XOM
2. A slightly fancier plot that compares the cumulative returns for 4 of the tickers, including SPY (the market)



In [ ]:
import matplotlib.pyplot as plt
#this is needed for the plotting

#Now plot cumulative return series for XOM
df_creturns["XOM"].plot()
#Even for basic plots, it is good practice to label the axes, so let's add a y-axis label
plt.ylabel("Cumulative Return")
plt.title('XOM Cumulative Return')

plt.show()

#The plot is looking reasonable -- note that the cumulative return falls sharply in 2020 due to Covid onset



In [ ]:
#Now plot cumulative return series for SPY and compare against several individual stocks
plt.plot(df_creturns['SPY'], label='SPY', linestyle='-')
plt.plot(df_creturns['CAT'], label='CAT', linestyle='--')
plt.plot(df_creturns['BA'], label='BA', linestyle=':')
plt.plot(df_creturns['XOM'], label='XOM', linestyle='-.')
#Even for basic plots, it is good practice to label the axes, so let's add a y-axis label
plt.ylabel("Cumulative Return")
plt.title('Cumulative Return Comparison')
plt.legend()

plt.show()

Some questions / insights based on the previous figures:

1. The cumulative return for XOM drops precipitously in early 2020 and then recovers. What explains this?
2. Somewhat related to the previous question, compare the behavior of BA relative to CAT and XOM as well as the market (SPY). What economic explanations are consistent with the cumulative return differences?
3. CAT outperforms SPY in terms of cumulative return over the sample period. What does the CAPM imply about the market portfolio as an investment portfolio? Do the results in this figure contradict the CAPM's implications regarding the market portfolio? Why or why not?



## Statistical Properties of Stock returns

Now that we have constructed a dataset of daily stock returns for 8 companies, we are ready to compute some basic statistics and gain insight regarding some of the properties of stock returns. 

As a first cut, let's look at the simple 'off the shelf' summary statistics produced by Pandas 'describe' functionality. Note that prior to computing the statistics I re-scale the returns to be in percentage units.

This 'off-the-shelf' application of .describe() gives the number of observations (count), mean, standard deviation (std), min, max, and several "quantiles" of the distribution of returns (25th, 50th, and 75th percentiles).

Questions:

1. What are the units for the mean and standard deviation? How might you further transform these to obtain more meaningful or interpretable numbers?
2. The minimum and maximum returns are useful for detecting potential data problems. Do these values seem reasonable to you?

In [16]:
df_returns_percent = 100*df_returns
df_returns_percent.describe()

TICKER,AAPL,BA,CAT,GE,IBM,SPY,TSLA,XOM
count,2516.000000,2516.000000,2516.000000,2516.000000,2516.000000,2516.000000,2516.000000,2516.000000
mean,0.108365,0.049860,0.082941,0.043264,0.042715,0.054796,0.195942,0.038689
std,1.792648,2.537197,1.874293,2.205622,1.500110,1.109570,3.600289,1.749974
min,-12.864700,-23.848400,-14.282200,-15.159200,-12.850700,-10.942400,-21.062800,-12.224800
25%,-0.734700,-1.003825,-0.867625,-0.940000,-0.636550,-0.370350,-1.616800,-0.829100
50%,0.099400,0.039350,0.064500,0.000000,0.076850,0.059600,0.126150,0.017100
75%,1.014050,1.098900,1.049800,0.990125,0.740750,0.592625,1.925525,0.886625
max,11.980800,24.318600,10.332100,14.730000,11.301100,9.060300,21.919000,12.686800



In this case, we do not see any "crazy" returns and the statistics seem reasonable.

### Key Insights

1. The mean return is positive for all 8 stocks and ranges from around 4 basis points per day (XOM,IBM) to 19 basis points per day (TSLA).

2. The standard deviation of daily returns for individual stocks ranges from 1.5 percent (IBM) to 3.6 percent (TSLA). Note that the standard deviation of daily returns is much larger than the mean for all of the stocks. This captures the idea that stock market fluctuations over very short horizon are very difficult to forecast. We expect this in informationally efficient asset markets (we will discuss this concept in more detail later.) Also notice that the SPY index has a smaller standard deviation (around 1.1%) than all of the individual stocks. This reflects the effects of diversification (we will talk about this in detail next week). 

3. The median return (50th percentile) is often, but not always, smaller than the mean. This suggests that at least some individual stock returns could be *positively skewed* (e.g., TSLA).

4. The "typical" daily return is small in magnitude; however, extreme price movements of over 10 percent in one day occur for nearly all of the stocks.

Although the .describe() functionality is useful, we now want to compute some additional statistics, beyond those reported in the above table, and produce a custom table that summarizes the key information and insights. 

## "Moments" of a random variable and corresponding statistics

Up to this point, we have been relatively informal regarding probability and statistics, relying on common and intuitive statistics and their interpretations. But, it is time to get a bit more formal about this. Therefore, let us remind ourselves of some basic concepts in probability. *Note: Here I assume familiarity with some probability ideas including random variables, distribution functions, density functions, etc. If you need a review, consult the textbook from your probability and statistics class.*

### Mean and Variance/Std deviation

The **expected value** or **mean** of a random variable is the average value. Mathematically, $E(X) = \int_{-\infty}^{\infty} x f(x) dx$, where $f(x)$ is the density function for the random variable (in the case of a discrete random variable the integral becomes a sum over possibly values and f(x) is the probability mass function). Sometimes we use the symbol $\mu$ for the mean or expectation, e.g., $\mu_{X}$ denotes the mean of the random variable $X$. 

The mean of a distribution is sometimes called the **first moment**. It is a measure of the "center of mass" of the distribution.  

The key "sample moments" for returns (or any data series) is the sample mean:

Mean or sample average: $\bar{R} \equiv (1/T)\sum_{t=1}^{T} R_{t}$


The **variance** of a random variable is the expected value of the squared deviation of the variable from its mean. Mathematically, $Var(X) = \sigma^{2}(X) = E[(X-\mu_{x})^{2}]=\int_{-\infty}^{\infty} (x-\mu_{x})^{2} f(x) dx$, where $\mu_{X}$ is the mean of $X$. 

The variance is the **second (centered) moment** of a random variable. It is a measure of the *dispersion* of a random variable. The square root of the variance is the standard deviation: $\sigma(X) = \sqrt{\sigma^{2}(X)}$. The standard deviation is in the same units as the underlying variable, e.g., monthly \% for returns, which makes it easier to interpret relative to the variance.

Similarly to the mean, we can construct a natural "plug-in" estimate of the variance: $s^2 \equiv (1/T)\sum_{t=1}^{T} (R_{t} - \bar{R})^{2}$.

*Technical note*: in contrast to the sample mean, the sample variance defined above is actually slightly biased even for i.i.d. returns. To obtain an unbiased variance estimate, we can divide the sum of squared deviations by $T-1$ rather than $T$, which represents a "degree of freedom" correction reflecting the fact that we used the data once to obtain an estimate of the (unknown) true mean.

### Higher moments (Skew and Kurtosis)

We can also consider other **higher moments**. The most common two are the **skewness (or skew)** and **kurtosis**, which are the third and fourth centered moments, respectively.

$\text{Skew}(X) =  \frac{E[(X-\mu_{x})^{3}]}{\sigma(X)^{3}}$. The skew is a measure of the degree of *asymmetry* in a distribution. The normal distribution has a skew of zero -- it is a symmetric distribution. 

As a real-life example of a skewed random variable, consider the payoff to the state lottery. This random variable is (positively) skewed: with high probability there is a small loss, but with a very small probability there is an extremely large gain.

$\text{Kurt}(X)  = \frac{E[(X-\mu_{x})^{4}]}{\sigma(X)^{4}}$. The kurtosis is a measure of "fatness of the tails" of a distribution. The normal distribution is a key reference. It has a kurtosis equal to 3. Distributions with a kurtosis higher than 3, such as "student's-$t$" distribution, are considered "fat-tailed". Under such distributions, extreme outcomes occur with higher probability relative to the normal distribution. 

The above moments are "population" features -- they describe the true distribution. For example, if we *know* that a particular random variable is normally distributed, then its skewness equals zero and its kurtosis equals 3. But what if instead we *observe data from an unknown distribution*? This case is generally more realistic. In such cases, we use observed data to *estimate* the unknown population moments.



(Sample) skewness: $\widehat{\text{Skew}} \equiv \frac{\frac{1}{T}\sum_{t=1}^{T} (R_{t} - \bar{R})^{3}} {(s^{2})^{3/2}}$

(Sample) kurtosis: $\widehat{\text{Kurt}} \equiv \frac{\frac{1}{T}\sum_{t=1}^{T} (R_{t} - \bar{R})^{4}} {(s^{2})^{2}}$

The above estimates take a relatively simple "plug-in" form. Basically, when we do not know a quantity, we "plug-in" the sample analog of the quantity. For example, when we do not know the true variance, we "plug-in" the sample variance.


Pandas allows a simple way to compute these statistics individually. Suppose, for example, that we want the sample skewness for each of the return series. Then we can use the ".skew()" method in Python:


In [14]:
df_returns.skew()

TICKER
AAPL   -0.002420
BA      0.203243
CAT    -0.159204
GE      0.142466
IBM    -0.420064
SPY    -0.553740
TSLA    0.275796
XOM     0.080582
dtype: float64

The skewness estimate is relatively small for most of the individual stocks, although IBM returns appear to be negatively skewed and TSLA returns appear positively skewed.  The estimated skew for the S\&P 500 is negative. 

*The homework assignment that follows this notebook will push further and teach you how to construct standard errors (measures of precision) for these skew estimates and to test the null hypothesis that the true skewness equals zero.*

For the kurtosis, we can likewise do:

In [15]:
df_returns.kurt()

TICKER
AAPL     5.314922
BA      16.406986
CAT      4.246369
GE       6.246171
IBM      9.912875
SPY     12.759707
TSLA     4.396796
XOM      6.461893
dtype: float64

*Important note*: Pandas .kurt() function computes the kurtosis using "Fisher’s definition of kurtosis" -- this is basically the kurtosis formula above with 3 (the kurtosis of the normal distribution) *subtracted*. Sometimes this is called the "excess kurtosis", meaning in excess of that for the normal distribution.

Thus, the positive values should be interpreted as indicating that the returns are more "fat-tailed" than the normal distribution. This is a well-known result that holds across many assets. Financial returns tend to have "fat tails" and produce extreme outcomes more often than predicted under the normal distribution. The result is important in areas such as risk management and in the regulation of financial institutions. 


### Introduction to statistical inference: Inference for the sample mean

The statistics we have computed for our stocks are just that: *statistics*. The point I seek to emphasize is that these statistics are *estimates* of the true, unknown properties of the distribution. 

Let's take as a concrete example the sample mean $\bar{X}$. We view statistics such as the sample mean as random variables that follow some probability distribution. In certain assumptions, we can determine this distribution exactly. For example, suppose that we assume that a set of observations $X_{1}, X_{2}, ..., X_{T}$ correspond to independent draws from a normal distribution with mean $\mu$ and (known) variance $\sigma^{2}$. 

The sample mean equals $(1/T) \sum_{t=1}^{T} X_{t}$. Using the fact that the sum of a series of normally distributed variables is also normally distributed, as well as the fact that a constant times a normally distributed random variable is again normally distributed, we can deduce that *the sample mean in this case follows a normal distribution*. 

In order to fully characterize the distribution of the sample mean in this setting, we need to determine the mean and variance of the distribution. 

First, we have that $E[(1/T) \sum_{t=1}^{T} X_{t}] = (1/T)\sum_{t=1}^{T} E(X_{t}) = (1/T) \sum_{t=1}^{T} \mu = \mu$. (Here we are using the fact that the expectation operator is *linear*.) In words, our result says that *the average value of the sample mean is the true mean*. In statistical parlance, the sample mean is an **unbiased estimator of the true mean**.

Next, we use the *independence* of the $X_{t}$ draws (we assumed this) to conclude that the variance of the sum of the $X_{t}$ equals the sum of the variances. (Note that this is not generally true when random variables are not independent.) In particular, we have that: 

$Var(\bar{X}) = (1/T^{2}) \sum_{t=1}^{t} Var(X_{t}) = (1/T^{2}) \sum_{t=1}^{T} \sigma^{2} = \frac{T \sigma^{2}}{T^{2}} = \frac{\sigma^{2}}{T}$

Thus, we have concluded that $\bar{X} \sim \text{Normal}(\mu,\sigma^{2}/T)$, which in words says that the sample mean follows a normal distribution with a mean equal to $\mu$ and a variance that equals $\sigma^{2}/T$, and therefore a standard deviation that equals $\sigma/\sqrt{T}$. 

*Was it important that the data were normally distributed?* We actually only used the assumption that $X_{t}$ are normally distributed once, when we asserted that the sample mean also follows a normal distribution (exactly). Even if $X_{t}$ is *not* normally distributed, i.e., it instead follows some other arbitrary distribution $D$ with mean $\mu$ and variance $\sigma$, then it remains true that the sample mean is unbiased ($E(\bar{X}) = \mu$) and the variance of the sample mean remains $\sigma/\sqrt{T}$. This is important, because it means that the sample mean has nice properties (unbiased, and a variance that shrinks toward zero at rate $\sqrt{T}$) quite generally and not just for normally distributed random variables. (We do need the i.i.d. assumption, though.) 

Notice that the variance goes toward zero as the sample size $T$ gets large. This makes sense. Intuitively, as we compute the mean using larger and larger samples of data, we obtain estimates that are very close to the true mean of $\mu$ with high probability. 

How could we use this result to do inference? Suppose we want a 95\% confidence interval for the mean $\mu$. Using the result above, this confidence interval can be computed as $[\bar{X} - 1.96 \times \sigma, \bar{X} + 1.96 \times \sigma]$. More generally, we can use the result:
$\frac{\sqrt{T}}{\sigma} (\bar{X} - \mu) \sim \text{Normal}(0,1)$, i.e., the re-centered and scaled sample mean is distributed as a standard normal random variable. 

*Note*: This result is exact if the data are i.i.d. normally distributed. But, even if the data follow some other distribution, even one that is skewed, for example, then the *central limit theorem* tells us that as the sample size gets large, i.e., as $T \rightarrow \infty$, then the distribution of the sample mean will still be approximately normally distributed.

To see how this result can be used, suppose we want to test the null hypothesis that $\mu=0$. Then, we can compare our **test statistic** (the scaled sample mean above) with the critical values of the standard normal distribution and *reject the null* if the absolute value of the test statistic exceeds a threshold that depends on the significance of the test.

*But there is a problem with these inference results*: we have assumed that we know the true parameter $\sigma^{2}$, and this is unrealistic in practice. 

A natural solution to this problem is to apply what statisticians sometimes call the "plug-in principle": we 'plug-in' a natural estimate of the unknown true standard deviation, i.e., we swap in $s^{2}$ for $\sigma^{2}$, where $s^{2}$ is the sample variance.

So, in the realistic case of an unknown variance, an estimate of the variance of the sample mean equals $\frac{s^{2}}{T}$ and an estimate of the standard deviation of the sample mean equals $\frac{\sqrt{s^{2}}}{\sqrt{T}}$. 

This substitution creates an issue, though. The statistic $\frac{\sqrt{T}}{s} (\bar{X}-\mu)$ does *not* exactly follow the standard normal distribution. Instead, it turns out that a slightly adjusted statistic follows the Student-$t$ distribution with $T-1$ degrees of freedom:
$\frac{\sqrt{T-1}}{s} (\bar{X} - \mu) \sim \mathcal{t}_{T-1}$, where $\mathcal{t}_{T-1}$ means the Student-$\mathcal{t}$ distribution with $T-1$ degrees of freedom. (*Note* the $\mathcal{t}$ here refers to the name of the distribution, not to be confused with an arbitrary sample observation $X_{t}$.)

The above distribution is exact for any $T > 1$ (under the given assumptions). In order to conduct statistical tests, one can look up quantiles of the $\mathcal{t}$-distribution in textbooks or in Python using the SciPy library (see scipy.stats.t).  However, as $T$ gets large, the Student's-$t$ distribution approaches the standard normal. For this reason, it is common in practice when the sample is relatively large (say $T > 40$) to simply use the standard normal reference distribution for inference. This point relates to the "central limit theorem" which posits that many statistics involving sample averages approximately (but not exactly) follow a normal distribution as the sample size gets large. We will talk more about this result in later classes.



### Constructing and presenting a (custom) summary statistics table

Earlier, we used Pandas .describe() method to produce a set of statistics for our stock return data. The statistics produced included the mean and standard deviation, but did not include estimates of the skewness and kurtosis. Now let us construct instead a "custom" statistics table that is also nicely formatted.

There are various ways to do this in Python. In this case, I will make use of the 'agg' method as a convenient way to get a set of custom statistics including sample skew and kurtosis. 

In addition to including the skewness and kurtosis estimates, we will apply our inference results above in order to compute the standard deviation or 'standard error' of the sample mean estimate. Finally, we will spit everything out in a reasonably nice table format.

In [17]:
summarydf = df_returns_percent.agg(['min','max','mean','median','std','skew','kurt','count'])
#print(type(summarydf))

#Now let's use our statistical results to compute the standard deviation for our estimates of the stock means

#This just pulls the sample size from the 'count' statistic results.
T = summarydf.iloc[-1,-1]

#Compute standard deviation of the means
stdevs = pd.DataFrame(summarydf.loc["std"]*(1/np.sqrt(T)))
print(type(stdevs))
print(stdevs)

#Here we are transposing the column of standard deviations of the mean to a row and adding it to our stats dataframe
summarydf2 = pd.concat([summarydf,np.transpose(stdevs)])
# Change the row indexes
summarydf2.index = ['Min', 'Max', 'Mean', 'Median','Std. Dev.','Skew','Kurt','N','Std. error of Mean']
#Finally, change row order so Std. of Mean appears right below Mean
summarydf3 = summarydf2.reindex(['Min', 'Max', 'Mean' ,'Std. error of Mean', 'Median','Std. Dev.','Skew','Kurt','N'])
print(summarydf3)

<class 'pandas.core.frame.DataFrame'>
             std
TICKER          
AAPL    0.035739
BA      0.050582
CAT     0.037366
GE      0.043972
IBM     0.029907
SPY     0.022121
TSLA    0.071776
XOM     0.034888
TICKER                     AAPL           BA          CAT           GE  \
Min                  -12.864700   -23.848400   -14.282200   -15.159200   
Max                   11.980800    24.318600    10.332100    14.730000   
Mean                   0.108365     0.049860     0.082941     0.043264   
Std. error of Mean     0.035739     0.050582     0.037366     0.043972   
Median                 0.099400     0.039350     0.064500     0.000000   
Std. Dev.              1.792648     2.537197     1.874293     2.205622   
Skew                  -0.002420     0.203243    -0.159204     0.142466   
Kurt                   5.314922    16.406986     4.246369     6.246171   
N                   2516.000000  2516.000000  2516.000000  2516.000000   

TICKER                      IBM          SPY       

The above is a reasonably good tabulation of summary statistics. But it could be improved for presentation. There is no title, for example. In addition, there are arguably too many decimal places displayed. If we want to include this table in a presentation or paper, we should reduce the precision to make it easier for the reader to glean the important information. 

One option would be to export this dataframe into, e.g., Excel or Word and then make additional formatting edits by hand. But that is inefficient and brings potential errors into play. Let's instead use Pandas 'styling' capabilities for dataframes. The following code sets the decimal precision at 3 and adds a descriptive title/caption to help the reader know exactly what is being presented.


In [ ]:
summarydf3.style.format('{0:,.3f}').set_caption("Summary Statistics for Daily Stock Returns, 2015-2024. Note: Kurtosis is reported in excess of 3.")

### More Insights and Questions

What can we learn from the above table about the properties of stock returns, beyond our earlier insights?

1. The skew statistics are mixed across the stocks, suggesting that some stocks have positively skewed returns while others are negatively skewed. This result is actually interesting and to some extent a feature of our relatively short, recent sample of data. Over a longer history, individual stock returns tend to be positively skewed.
2. The stock market proxy (SPY) has negatively skewed returns. This is consistent with results using much longer historical samples of data.
3. All of the (excess) kurtosis statistics are positive, and many of them are large. There is, therefore, strong evidence that the distribution of daily stock returns is *fat-tailed*, i.e., extreme outcomes occur more frequently than predicted by the normal distribution. 
4. The standard deviation of the mean is often similar in magnitude to the mean estimate itself. This means that we have relatively little precision with respect to our estimates of the means. For example, the mean estimate for IBM is 0.04 (% per day). But the standard deviation of this estimate is close to 0.03%. It is easy to see that a standard confidence interval would *include zero*, i.e., we cannot statistically reject the null that the true mean of IBM is zero. 

**Questions** (for you):

1. Is it appropriate to conclude that CAT returns are negatively skewed based on the table above? Explain your reasoning.

2. Consider the S&P 500 ETF (SPY). Can we reject the null that the mean return on this security is zero? Explain your reasoning. What would economic theory suggest regarding this mean? Again, explain.



